<a href="https://colab.research.google.com/github/guirco/ufpel-pdi-2026-1/blob/main/2026_1%20_LAB_Outros_Tipos_Sinais_Visuais_Nuvem_Malha3D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade de Laboratório: Reconstrução de Malha 3D a partir de Nuvem de Pontos

**Tema:** pré-processamento, remoção de ruído e reconstrução de malha 3D usando Python e Open3D no Google Colab.

Nesta atividade, vocês irão transformar uma nuvem de pontos 3D em uma malha triangular. A atividade foi dividida em fases para ser executada aos poucos durante a aula prática.

**Faça o donwload da nuvem de pontos (SantaBustoMadeira_2.ply), por meio deste** [link do Google Drive](https://drive.google.com/drive/folders/1umYFE148TS4eHKe2AvuH9dC2o6O2j4DC?usp=sharing), e coloque ela na pasta abaixo:

```text
/content
```

Formatos aceitos neste notebook:

```text
.ply, .pcd, .xyz, .xyzn, .xyzrgb, .pts
```

Ao final, serão gerados arquivos de saída em:

```text
/content/resultados_malha
```

## Objetivos da atividade

Ao final desta prática, o aluno deverá ser capaz de:

1. carregar uma nuvem de pontos 3D no Colab;
2. inspecionar informações básicas da nuvem, como quantidade de pontos e caixa delimitadora;
3. aplicar técnicas de pré-processamento antes da reconstrução;
4. remover ruídos e pontos isolados da nuvem;
5. estimar normais da superfície;
6. gerar uma malha 3D usando o algoritmo **Ball Pivoting Algorithm**, BPA;
7. opcionalmente, comparar com a reconstrução por **Poisson Surface Reconstruction**;
8. salvar a malha final em formato `.ply`.

# Fase 1 — Preparação do ambiente

Execute esta célula primeiro para instalar e importar as bibliotecas necessárias.

> No Colab, a instalação do Open3D pode levar alguns minutos na primeira execução.

In [ ]:
!pip install -q open3d plotly

import os
import glob
import numpy as np
import open3d as o3d

from pathlib import Path

print("Open3D:", o3d.__version__)
print("Ambiente preparado com sucesso.")

# Fase 2 — Configuração das pastas e localização da nuvem de pontos

Nesta fase, o notebook procura automaticamente um arquivo de nuvem de pontos dentro de `/content`.

Caso existam vários arquivos, o primeiro encontrado será usado. Se desejar selecionar manualmente, altere a variável `arquivo_manual`.

In [ ]:
# Pasta onde a nuvem de pontos estará no Colab
input_path = "/content"

# Pasta onde serão salvos os resultados
output_path = "/content/resultados_malha"
os.makedirs(output_path, exist_ok=True)

# Extensões aceitas
extensoes = ["*.ply", "*.pcd", "*.xyz", "*.xyzn", "*.xyzrgb", "*.pts"]

# Se quiser escolher manualmente um arquivo, coloque o caminho aqui.
# Exemplo: arquivo_manual = "/content/minha_nuvem.ply"
arquivo_manual = ""

if arquivo_manual.strip():
    full_input_path = arquivo_manual
else:
    arquivos_encontrados = []
    for ext in extensoes:
        arquivos_encontrados.extend(glob.glob(os.path.join(input_path, ext)))

    arquivos_encontrados = sorted(arquivos_encontrados)

    if len(arquivos_encontrados) == 0:
        raise FileNotFoundError(
            "Nenhum arquivo de nuvem de pontos foi encontrado em /content. "
            "Faça upload de um arquivo .ply, .pcd, .xyz, .xyzn, .xyzrgb ou .pts."
        )

    full_input_path = arquivos_encontrados[0]

nome_arquivo = os.path.basename(full_input_path)
nome_base = Path(nome_arquivo).stem

print("Arquivo selecionado:", full_input_path)
print("Nome base:", nome_base)
print("Pasta de saída:", output_path)

# Fase 3 — Leitura da nuvem de pontos

Agora vamos carregar a nuvem de pontos usando o Open3D.

Nesta etapa, ainda **não** fazemos reconstrução. Primeiro precisamos verificar se os dados foram carregados corretamente.

**Obs.:** Os normais de uma nuvem de pontos são vetores matemáticos perpendiculares à superfície local de um objeto em cada ponto específico.

In [ ]:
pcd_original = o3d.io.read_point_cloud(full_input_path)

if not pcd_original.has_points():
    raise ValueError(
        "A nuvem foi lida, mas não contém pontos. "
        "Verifique se o arquivo está correto ou se o formato é suportado."
    )

print("Nuvem carregada com sucesso.")
print("Quantidade de pontos:", np.asarray(pcd_original.points).shape[0])
print("Possui cores?", pcd_original.has_colors())
print("Possui normais?", pcd_original.has_normals())

# Fase 4 — Diagnóstico inicial da nuvem

Antes de limpar ou reconstruir a malha, é importante observar algumas propriedades da nuvem:

- quantidade de pontos;
- dimensões da caixa delimitadora;
- centro da nuvem;
- presença de cor RGB;
- presença de normais.

Essas informações ajudam a escolher parâmetros como `voxel_size`, `radius` e `nb_neighbors`.

In [ ]:
def relatorio_nuvem(pcd, nome="Nuvem"):
    pontos = np.asarray(pcd.points)
    print("=" * 60)
    print(nome)
    print("=" * 60)
    print("Quantidade de pontos:", len(pontos))
    print("Possui cores?", pcd.has_colors())
    print("Possui normais?", pcd.has_normals())

    if len(pontos) > 0:
        aabb = pcd.get_axis_aligned_bounding_box()
        min_bound = aabb.get_min_bound()
        max_bound = aabb.get_max_bound()
        extent = aabb.get_extent()
        centro = aabb.get_center()

        print("Limite mínimo (x, y, z):", min_bound)
        print("Limite máximo (x, y, z):", max_bound)
        print("Dimensões da caixa (x, y, z):", extent)
        print("Centro aproximado:", centro)
        print("Maior dimensão:", np.max(extent))

relatorio_nuvem(pcd_original, "Nuvem original")

## Visualização simples da nuvem

A visualização 3D interativa pode variar conforme o ambiente do Colab. Por isso, usamos uma visualização simples por amostragem com `matplotlib`.

> Esta visualização é apenas para inspeção. A reconstrução usará a nuvem completa ou a nuvem pré-processada.

In [ ]:
import matplotlib.pyplot as plt


def visualizar_amostra_pontos(pcd, titulo="Amostra da nuvem de pontos", max_pontos=8000):
    pontos = np.asarray(pcd.points)
    if len(pontos) == 0:
        print("Nuvem vazia.")
        return

    n = min(max_pontos, len(pontos))
    idx = np.random.choice(len(pontos), n, replace=False)
    amostra = pontos[idx]

    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(amostra[:, 0], amostra[:, 1], amostra[:, 2], s=0.5)
    ax.set_title(titulo)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    plt.show()

visualizar_amostra_pontos(pcd_original, "Nuvem original - amostra")

# Fase 5 — Pré-processamento e remoção de ruído

Esta é a etapa mais importante antes de transformar a nuvem em malha.

Uma nuvem de pontos capturada por sensores, reconstruída por fotogrametria ou extraída de datasets pode conter:

- pontos duplicados;
- pontos com valores inválidos;
- pontos isolados distantes da superfície principal;
- ruído local;
- densidade irregular;
- escala inadequada para alguns algoritmos.

Nesta fase, serão aplicados os seguintes procedimentos:

1. remoção de pontos inválidos, como `NaN` e `Inf`;
2. remoção de pontos duplicados;
3. centralização opcional da nuvem;
4. redução de densidade por **voxel downsampling**;
5. remoção de ruído por **Statistical Outlier Removal**, SOR;
6. remoção de ruído por **Radius Outlier Removal**, ROR;
7. estimativa e orientação de normais.

In [ ]:
# ============================================================
# Parâmetros didáticos de pré-processamento
# ============================================================

# Ativa ou desativa a centralização da nuvem no centro do sistema de coordenadas.
# Por padrão, fica False para preservar as coordenadas originais da nuvem.
centralizar_nuvem = False

# Voxel downsampling:
# - valores menores preservam mais detalhes;
# - valores maiores reduzem mais pontos, mas podem perder detalhes.
# Use None para desativar.
voxel_size = None

# Remoção estatística de outliers, SOR:
# nb_neighbors: número de vizinhos analisados por ponto.
# std_ratio: quanto menor, mais agressiva é a remoção.
aplicar_sor = True
sor_nb_neighbors = 30
sor_std_ratio = 2.0

# Remoção por raio, ROR:
# remove pontos que não possuem vizinhos suficientes dentro de um determinado raio.
aplicar_ror = True
ror_nb_points = 8
ror_radius_factor = 2.5

# Estimativa de normais:
normal_radius_factor = 4.0
normal_max_nn = 40
orientar_normais = True
orientar_k = 20

# evita que uma configuração muito agressiva deixe a nuvem quase vazia.
min_pontos_apos_limpeza = 100

In [ ]:
def remover_pontos_invalidos(pcd):
    """Remove pontos com NaN ou Inf, preservando cores e normais quando existirem."""
    pontos = np.asarray(pcd.points)
    mascara = np.all(np.isfinite(pontos), axis=1)

    novo = o3d.geometry.PointCloud()
    novo.points = o3d.utility.Vector3dVector(pontos[mascara])

    if pcd.has_colors():
        cores = np.asarray(pcd.colors)
        novo.colors = o3d.utility.Vector3dVector(cores[mascara])

    if pcd.has_normals():
        normais = np.asarray(pcd.normals)
        novo.normals = o3d.utility.Vector3dVector(normais[mascara])

    return novo


def distancia_media_vizinhos(pcd):
    """Calcula uma distância típica entre pontos usando vizinho mais próximo."""
    distancias = np.asarray(pcd.compute_nearest_neighbor_distance())
    distancias = distancias[np.isfinite(distancias)]
    distancias = distancias[distancias > 0]

    if len(distancias) == 0:
        return 0.01

    # A mediana é mais robusta do que a média quando há ruído/outliers.
    return float(np.median(distancias))


def preprocessar_nuvem(
    pcd,
    centralizar=True,
    voxel_size=None,
    aplicar_sor=True,
    sor_nb_neighbors=30,
    sor_std_ratio=2.0,
    aplicar_ror=True,
    ror_nb_points=8,
    ror_radius_factor=2.5,
    normal_radius_factor=4.0,
    normal_max_nn=40,
    orientar_normais=True,
    orientar_k=20,
    min_pontos_apos_limpeza=100
):
    """Aplica limpeza, remoção de ruído e estimação de normais."""

    print("Iniciando pré-processamento...")
    pcd_limpa = o3d.geometry.PointCloud(pcd)
    print("Pontos iniciais:", len(pcd_limpa.points))

    # 1) Remover NaN e Inf
    pcd_limpa = remover_pontos_invalidos(pcd_limpa)
    print("Após remover NaN/Inf:", len(pcd_limpa.points))

    # 2) Remover pontos duplicados, se a versão do Open3D suportar o método.
    if hasattr(pcd_limpa, "remove_duplicated_points"):
        pcd_limpa = pcd_limpa.remove_duplicated_points()
        print("Após remover duplicados:", len(pcd_limpa.points))

    # 3) Centralizar a nuvem
    if centralizar:
        centro = pcd_limpa.get_center()
        pcd_limpa.translate(-centro)
        print("Nuvem centralizada no sistema de coordenadas.")

    # 4) Distância típica entre pontos
    d_mediana = distancia_media_vizinhos(pcd_limpa)
    print(f"Distância mediana entre vizinhos: {d_mediana:.6f}")

    # 5) Voxel downsampling
    if voxel_size is not None and voxel_size > 0:
        pcd_limpa = pcd_limpa.voxel_down_sample(voxel_size=voxel_size)
        print(f"Após voxel downsampling ({voxel_size}):", len(pcd_limpa.points))

    # 6) Statistical Outlier Removal, SOR
    if aplicar_sor and len(pcd_limpa.points) > sor_nb_neighbors:
        pcd_antes_sor = o3d.geometry.PointCloud(pcd_limpa)
        pcd_sor, ind = pcd_limpa.remove_statistical_outlier(
            nb_neighbors=sor_nb_neighbors,
            std_ratio=sor_std_ratio
        )
        if len(pcd_sor.points) >= min_pontos_apos_limpeza:
            pcd_limpa = pcd_sor
            print("Após SOR:", len(pcd_limpa.points))
        else:
            pcd_limpa = pcd_antes_sor
            print("SOR foi agressivo demais e foi revertido. Pontos mantidos:", len(pcd_limpa.points))

    # 7) Radius Outlier Removal, ROR
    if aplicar_ror and len(pcd_limpa.points) > ror_nb_points:
        pcd_antes_ror = o3d.geometry.PointCloud(pcd_limpa)
        d_mediana = distancia_media_vizinhos(pcd_limpa)
        ror_radius = max(d_mediana * ror_radius_factor, 1e-6)
        pcd_ror, ind = pcd_limpa.remove_radius_outlier(
            nb_points=ror_nb_points,
            radius=ror_radius
        )
        if len(pcd_ror.points) >= min_pontos_apos_limpeza:
            pcd_limpa = pcd_ror
            print(f"Após ROR, raio={ror_radius:.6f}:", len(pcd_limpa.points))
        else:
            pcd_limpa = pcd_antes_ror
            print("ROR foi agressivo demais e foi revertido. Pontos mantidos:", len(pcd_limpa.points))

    # 8) Estimar normais
    d_mediana = distancia_media_vizinhos(pcd_limpa)
    normal_radius = max(d_mediana * normal_radius_factor, 1e-6)

    pcd_limpa.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(
            radius=normal_radius,
            max_nn=normal_max_nn
        )
    )
    pcd_limpa.normalize_normals()
    print(f"Normais estimadas com raio={normal_radius:.6f} e max_nn={normal_max_nn}.")

    # 9) Orientar normais de forma consistente
    if orientar_normais and len(pcd_limpa.points) > orientar_k:
        try:
            pcd_limpa.orient_normals_consistent_tangent_plane(k=orientar_k)
            print("Normais orientadas de forma consistente.")
        except Exception as erro:
            print("Aviso: não foi possível orientar as normais de forma consistente.")
            print("Motivo:", erro)

    print("Pré-processamento concluído.")
    return pcd_limpa


pcd_preprocessada = preprocessar_nuvem(
    pcd_original,
    centralizar=centralizar_nuvem,
    voxel_size=voxel_size,
    aplicar_sor=aplicar_sor,
    sor_nb_neighbors=sor_nb_neighbors,
    sor_std_ratio=sor_std_ratio,
    aplicar_ror=aplicar_ror,
    ror_nb_points=ror_nb_points,
    ror_radius_factor=ror_radius_factor,
    normal_radius_factor=normal_radius_factor,
    normal_max_nn=normal_max_nn,
    orientar_normais=orientar_normais,
    orientar_k=orientar_k,
    min_pontos_apos_limpeza=min_pontos_apos_limpeza
)

relatorio_nuvem(pcd_preprocessada, "Nuvem pré-processada")

## Comparação visual: antes e depois do pré-processamento

Observe se a remoção de ruído preservou o formato principal do objeto.

Se muitos detalhes sumirem, tente:

- aumentar `sor_std_ratio`;
- reduzir `ror_nb_points`;
- aumentar `ror_radius_factor`;
- desativar temporariamente o `voxel_size`;
- reduzir o valor de `voxel_size`, caso ele esteja ativado.

In [ ]:
visualizar_amostra_pontos(pcd_original, "Antes do pré-processamento")
visualizar_amostra_pontos(pcd_preprocessada, "Depois do pré-processamento")

# Fase 6 — Salvamento da nuvem pré-processada

Antes de gerar a malha, vamos salvar a nuvem limpa.

Assim, você pode comparar:

- arquivo original;
- arquivo pré-processado;
- malha reconstruída.

In [ ]:
caminho_pcd_limpa = os.path.join(output_path, f"{nome_base}_preprocessada.ply")
o3d.io.write_point_cloud(caminho_pcd_limpa, pcd_preprocessada)
print("Nuvem pré-processada salva em:", caminho_pcd_limpa)

# Fase 7 — Reconstrução de malha com Ball Pivoting Algorithm, BPA

O **Ball Pivoting Algorithm** reconstrói uma malha triangular girando uma esfera virtual entre pontos próximos.

Ele funciona melhor quando:

- a nuvem tem boa densidade;
- os pontos estão bem distribuídos;
- as normais estão estimadas corretamente;
- o raio da esfera é compatível com a distância entre os pontos.

Aqui, o raio é calculado automaticamente a partir da distância mediana entre vizinhos.

In [ ]:
def reconstruir_bpa(pcd, multiplicadores_raio=(2.0, 3.0, 4.0)):
    """Reconstrói uma malha usando Ball Pivoting Algorithm."""

    if not pcd.has_normals():
        raise ValueError("A nuvem precisa ter normais estimadas antes do BPA.")

    d_mediana = distancia_media_vizinhos(pcd)
    raios = [max(d_mediana * m, 1e-6) for m in multiplicadores_raio]

    print("Distância mediana entre pontos:", d_mediana)
    print("Raios usados no BPA:", raios)

    mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
        pcd,
        o3d.utility.DoubleVector(raios)
    )

    return mesh


bpa_mesh = reconstruir_bpa(pcd_preprocessada, multiplicadores_raio=(2.0, 3.0, 4.0))

print("Malha BPA criada.")
print("Vértices:", len(bpa_mesh.vertices))
print("Triângulos:", len(bpa_mesh.triangles))

# Fase 8 — Limpeza e simplificação da malha

Depois da reconstrução, é comum que a malha tenha triângulos degenerados, vértices duplicados ou arestas não-manifold.

Nesta etapa, fazemos a limpeza geométrica da malha. Também é possível simplificar a malha para reduzir o número de triângulos.

In [ ]:
def limpar_malha(mesh):
    """Remove problemas geométricos comuns da malha."""
    mesh_limpa = o3d.geometry.TriangleMesh(mesh)

    mesh_limpa.remove_degenerate_triangles()
    mesh_limpa.remove_duplicated_triangles()
    mesh_limpa.remove_duplicated_vertices()
    mesh_limpa.remove_non_manifold_edges()
    mesh_limpa.compute_vertex_normals()

    return mesh_limpa


def simplificar_malha(mesh, numero_alvo_triangulos=None):
    """Simplifica a malha, se um número alvo de triângulos for definido."""
    if numero_alvo_triangulos is None:
        return mesh

    numero_atual = len(mesh.triangles)
    if numero_atual <= numero_alvo_triangulos:
        print("A malha já possui menos triângulos do que o alvo definido.")
        return mesh

    mesh_simplificada = mesh.simplify_quadric_decimation(numero_alvo_triangulos)
    mesh_simplificada = limpar_malha(mesh_simplificada)
    return mesh_simplificada


# Use None para não simplificar.
# Exemplo: numero_alvo_triangulos = 100000
numero_alvo_triangulos = None

bpa_mesh_limpa = limpar_malha(bpa_mesh)
bpa_mesh_final = simplificar_malha(bpa_mesh_limpa, numero_alvo_triangulos)

print("Malha final BPA:")
print("Vértices:", len(bpa_mesh_final.vertices))
print("Triângulos:", len(bpa_mesh_final.triangles))

## Transferência de cores da nuvem para a malha

Se a nuvem original possui cores RGB, podemos transferir a cor do ponto mais próximo para cada vértice da malha.

Isso ajuda a preservar a aparência visual do objeto após a reconstrução.

In [ ]:
def transferir_cores_ponto_para_malha(pcd, mesh):
    """Transfere cores da nuvem para os vértices da malha por vizinho mais próximo."""
    if not pcd.has_colors():
        print("A nuvem não possui cores. A malha será salva sem cores RGB.")
        return mesh

    pontos = np.asarray(pcd.points)
    cores = np.asarray(pcd.colors)
    vertices = np.asarray(mesh.vertices)

    if len(pontos) == 0 or len(vertices) == 0:
        return mesh

    pcd_tree = o3d.geometry.KDTreeFlann(pcd)
    cores_vertices = []

    for v in vertices:
        _, idx, _ = pcd_tree.search_knn_vector_3d(v, 1)
        cores_vertices.append(cores[idx[0]])

    mesh_colorida = o3d.geometry.TriangleMesh(mesh)
    mesh_colorida.vertex_colors = o3d.utility.Vector3dVector(np.asarray(cores_vertices))
    return mesh_colorida


bpa_mesh_final = transferir_cores_ponto_para_malha(pcd_preprocessada, bpa_mesh_final)
print("Possui cores nos vértices?", bpa_mesh_final.has_vertex_colors())

# Fase 9 — Salvamento da malha final

Agora salvamos a malha reconstruída em `.ply`.

O formato `.ply` é adequado porque pode armazenar geometria e cores nos vértices.

In [ ]:
caminho_bpa = os.path.join(output_path, f"{nome_base}_malha_BPA.ply")
o3d.io.write_triangle_mesh(caminho_bpa, bpa_mesh_final)

print("Malha BPA salva em:", caminho_bpa)

# Fase 10 — Visualização simples da malha

A visualização abaixo mostra uma amostra dos vértices da malha e algumas arestas/triângulos podem não aparecer de forma detalhada no Colab.

Para inspeção completa, baixe o arquivo `.ply` e abra em softwares como:

- MeshLab;
- CloudCompare;
- Blender;
- Open3D localmente.

In [ ]:
def visualizar_vertices_malha(mesh, titulo="Vértices da malha", max_vertices=8000):
    vertices = np.asarray(mesh.vertices)
    if len(vertices) == 0:
        print("Malha vazia.")
        return

    n = min(max_vertices, len(vertices))
    idx = np.random.choice(len(vertices), n, replace=False)
    amostra = vertices[idx]

    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(amostra[:, 0], amostra[:, 1], amostra[:, 2], s=0.5)
    ax.set_title(titulo)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    plt.show()

visualizar_vertices_malha(bpa_mesh_final, "Malha BPA - amostra dos vértices")

# Fase 11 — Reconstrução alternativa por Poisson, opcional

A reconstrução por **Poisson Surface Reconstruction** tende a gerar superfícies mais fechadas e contínuas.

Ela pode ser útil quando a nuvem possui boa orientação de normais. Porém, também pode criar superfícies artificiais em regiões onde não existem pontos.

Execute esta fase apenas depois de concluir o BPA.

In [ ]:
executar_poisson = False


def reconstruir_poisson(pcd, depth=8, quantile_remocao=0.02):
    """Reconstrói malha por Poisson e remove vértices de baixa densidade."""
    if not pcd.has_normals():
        raise ValueError("A nuvem precisa ter normais estimadas antes do Poisson.")

    print("Executando Poisson Surface Reconstruction...")
    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        pcd,
        depth=depth
    )

    densities = np.asarray(densities)
    limite = np.quantile(densities, quantile_remocao)
    vertices_remover = densities < limite
    mesh.remove_vertices_by_mask(vertices_remover)

    # Recorta a malha para a caixa da nuvem, reduzindo superfícies artificiais externas.
    bbox = pcd.get_axis_aligned_bounding_box()
    mesh = mesh.crop(bbox)
    mesh = limpar_malha(mesh)

    return mesh


if executar_poisson:
    poisson_mesh = reconstruir_poisson(pcd_preprocessada, depth=8, quantile_remocao=0.02)
    poisson_mesh = transferir_cores_ponto_para_malha(pcd_preprocessada, poisson_mesh)

    caminho_poisson = os.path.join(output_path, f"{nome_base}_malha_Poisson.ply")
    o3d.io.write_triangle_mesh(caminho_poisson, poisson_mesh)

    print("Malha Poisson salva em:", caminho_poisson)
    print("Vértices:", len(poisson_mesh.vertices))
    print("Triângulos:", len(poisson_mesh.triangles))
else:
    print("Poisson está desativado. Para executar, altere executar_poisson = True e rode novamente esta célula.")

# Fase 12 — Geração de diferentes níveis de detalhe, LoD

Esta etapa cria versões simplificadas da malha para comparação.

Isso é útil em aplicações de visualização 3D, realidade virtual e processamento em dispositivos com menor capacidade computacional.

In [ ]:
gerar_lods = True
lods = [100000, 50000, 20000, 10000]


def exportar_lods(mesh, lods, output_path, nome_base):
    arquivos = []
    for alvo in lods:
        if len(mesh.triangles) <= alvo:
            print(f"Ignorando LoD {alvo}: a malha possui apenas {len(mesh.triangles)} triângulos.")
            continue

        mesh_lod = mesh.simplify_quadric_decimation(alvo)
        mesh_lod = limpar_malha(mesh_lod)

        caminho = os.path.join(output_path, f"{nome_base}_BPA_lod_{alvo}.ply")
        o3d.io.write_triangle_mesh(caminho, mesh_lod)
        arquivos.append(caminho)
        print("LoD salvo:", caminho)

    return arquivos


if gerar_lods:
    arquivos_lod = exportar_lods(bpa_mesh_final, lods, output_path, nome_base)
else:
    print("Geração de LoDs desativada.")

# Fase 13 — Compactação dos resultados para download

Esta etapa compacta os arquivos gerados em um `.zip`.

In [ ]:
import shutil

zip_base = f"/content/{nome_base}_resultados_malha"
zip_path = shutil.make_archive(zip_base, "zip", output_path)

print("Arquivo compactado gerado em:", zip_path)

#EXERCÍCIOS — Atividade prática

## Parte A — Observação inicial

1. Quantos pontos existem na nuvem original?
2. A nuvem possui cores RGB?
3. A nuvem possui normais antes do pré-processamento?
4. Qual é a maior dimensão da caixa delimitadora?

## Parte B — Remoção de ruído

Altere os parâmetros abaixo e observe os resultados:

- `sor_std_ratio = 1.0`, `2.0`, `3.0`;
- `sor_nb_neighbors = 10`, `30`, `50`;
- `ror_nb_points = 4`, `8`, `16`;
- `ror_radius_factor = 1.5`, `2.5`, `4.0`.

Responda:

1. Qual configuração removeu mais pontos?
2. Qual configuração preservou melhor o formato do objeto?
3. Alguma configuração removeu detalhes importantes?

## Parte C — Reconstrução de malha

Altere os multiplicadores de raio do BPA:

```python
multiplicadores_raio=(1.5, 2.0, 2.5)
multiplicadores_raio=(2.0, 3.0, 4.0)
multiplicadores_raio=(3.0, 4.0, 5.0)
```

Responda:

1. Qual configuração gerou mais triângulos?
2. Qual configuração gerou menos buracos?
3. Qual configuração distorceu menos a forma original?

## Parte D — Comparação BPA versus Poisson

Ative:

```python
executar_poisson = True
```

Compare a malha Poisson com a malha BPA.

Responda:

1. Qual método gerou superfície mais fechada?
2. Qual método preservou melhor os detalhes locais?
3. O método Poisson criou partes artificiais na malha?

## Entrega sugerida

Cada grupo deve entregar:

1. a nuvem pré-processada `.ply`;
2. a malha BPA final `.ply`;
3. opcionalmente, a malha Poisson `.ply`;
4. relatar quais os parâmetros utilizados e uma comparação visual dos resultados.

# Resumo técnico da prática

Nesta atividade, a reconstrução não é feita diretamente sobre a nuvem original. Primeiro, a nuvem passa por uma etapa de pré-processamento para reduzir ruídos, corrigir pontos inválidos, remover duplicações, estimar normais e melhorar a distribuição dos pontos. Em seguida, a reconstrução por BPA utiliza a distância mediana entre vizinhos para definir raios adequados à escala da nuvem. Após a geração da malha, são aplicadas rotinas de limpeza geométrica e, quando houver cores nos pontos, elas são transferidas para os vértices da malha.